# 💬 Sentiment Analysis – Furniture Shop ML Service

Notebook này trình bày phương pháp **Sentiment Analysis** được triển khai trong `ml_service/app/services/sentiment_service.py`.

## Tổng quan

Hệ thống phân tích cảm xúc theo **rule-based approach** kết hợp:
- **Phân tích từ khóa** từ comment (positive/negative word matching)
- **Điểm đánh giá** (rating từ 1–5 sao)

Kết quả: Label **Good** / **Bad** + confidence score.

## Logic phân loại
```
if rating >= 4 and score >= -1  → "Good"
elif rating <= 2 and score <= 1 → "Bad"
elif score > 0                  → "Good"
else                            → "Bad"

confidence = min(1.0, 0.55 + |score|×0.15 + |rating-3.0|×0.05)
```

## 1. Thiết lập môi trường

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib', 'seaborn', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('✅ Thư viện đã sẵn sàng')

## 2. Nạp dữ liệu

In [ ]:
reviews_raw  = pd.read_csv('furniture_shop.reviews.csv')
products_raw = pd.read_csv('furniture_shop.products.csv')

print(f'⭐ Reviews : {len(reviews_raw)}')
print(f'📦 Products: {len(products_raw)}')
reviews_raw[['_id','product','user','rating','comment']].head()

## 3. Từ điển Positive/Negative Words

In [ ]:
POSITIVE_WORDS = {
    'tot', 'dep', 'hai long', 'chat luong', 'xuat sac',
    'yeu thich', 'recommend', 'good', 'great', 'excellent', 'worth'
}

NEGATIVE_WORDS = {
    'te', 'kem', 'that vong', 'khong hai long', 'vo',
    'xau', 'bad', 'poor', 'terrible', 'worst', 'not good'
}

print('✅ Positive words:', sorted(POSITIVE_WORDS))
print('❌ Negative words:', sorted(NEGATIVE_WORDS))

## 4. Hàm `sentiment_label()` – Core Logic

In [ ]:
def sentiment_label(comment: str, rating: float) -> dict:
    """
    Tái hiện chính xác hàm sentiment_label() từ sentiment_service.py
    """
    text  = (comment or '').lower()
    score = 0

    for token in POSITIVE_WORDS:
        if token in text:
            score += 1
    for token in NEGATIVE_WORDS:
        if token in text:
            score -= 1

    if rating >= 4 and score >= -1:
        label = 'Good'
    elif rating <= 2 and score <= 1:
        label = 'Bad'
    elif score > 0:
        label = 'Good'
    else:
        label = 'Bad'

    confidence = min(1.0, 0.55 + abs(score) * 0.15 + abs(rating - 3.0) * 0.05)
    return {'label': label, 'confidence': round(confidence, 3), 'score': score}

# Test cases
test_cases = [
    ('Rat hai long, dep hon mong doi.', 5.0),
    ('Not good as expected, can improve.', 2.0),
    ('San pham dung duoc nhung giao hoi cham.', 3.0),
    ('Khong hai long lam ve mau sac.', 3.0),
    ('Good quality, worth the money.', 4.0),
    ('Chat luong tam on, chua nhu ky vong.', 2.0),
]

print('📊 Ví dụ phân tích cảm xúc:')
print('-'*80)
results = []
for comment, rating in test_cases:
    r = sentiment_label(comment, rating)
    results.append({'comment': comment[:45], 'rating': rating, **r})
    icon = '✅' if r['label'] == 'Good' else '❌'
    print(f"{icon} [{r['label']:4s}] conf={r['confidence']:.3f} | rating={rating} | '{comment[:45]}'")

pd.DataFrame(results)

## 5. Hàm `sentiment_analysis()` – Phân tích toàn bộ Reviews

In [ ]:
def sentiment_analysis(reviews_df: pd.DataFrame) -> dict:
    """Tái hiện sentiment_analysis() từ sentiment_service.py"""
    labelled  = []
    good_count = 0
    bad_count  = 0

    for _, review in reviews_df.iterrows():
        result = sentiment_label(
            review.get('comment', ''),
            float(review.get('rating') or 0)
        )
        row = {
            'review_id' : str(review.get('_id', '')),
            'product_id': str(review.get('product', '')),
            'user_id'   : str(review.get('user', '')),
            'rating'    : float(review.get('rating') or 0),
            'comment'   : str(review.get('comment', '')),
            **result,
        }
        labelled.append(row)
        if result['label'] == 'Good':
            good_count += 1
        else:
            bad_count += 1

    total = len(labelled)
    return {
        'summary': {
            'total'     : total,
            'good'      : good_count,
            'bad'       : bad_count,
            'good_ratio': round(good_count / total, 4) if total else 0,
            'bad_ratio' : round(bad_count  / total, 4) if total else 0,
        },
        'reviews': labelled,
    }

result = sentiment_analysis(reviews_raw)
summary = result['summary']
print('📊 SENTIMENT ANALYSIS SUMMARY')
print('='*40)
print(f"  Tổng reviews : {summary['total']}")
print(f"  ✅ Good       : {summary['good']} ({summary['good_ratio']*100:.1f}%)")
print(f"  ❌ Bad        : {summary['bad']} ({summary['bad_ratio']*100:.1f}%)")

reviews_df = pd.DataFrame(result['reviews'])
reviews_df[['comment','rating','label','confidence','score']].head(8)

## 6. Visualisation – Kết quả

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Pie chart Good/Bad
ax1 = fig.add_subplot(gs[0, 0])
ax1.pie([summary['good'], summary['bad']],
        labels=[f"Good ({summary['good']})", f"Bad ({summary['bad']})"],
        colors=['#2ecc71', '#e74c3c'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'linewidth': 2, 'edgecolor': 'white'},
        textprops={'fontsize': 10})
ax1.set_title('Tỷ lệ Good/Bad', fontweight='bold')

# 2. Distribution by rating
ax2 = fig.add_subplot(gs[0, 1])
rating_good = reviews_df[reviews_df['label']=='Good']['rating'].value_counts().sort_index()
rating_bad  = reviews_df[reviews_df['label']=='Bad']['rating'].value_counts().sort_index()
x = np.arange(1, 6)
ax2.bar(x - 0.2, [rating_good.get(r, 0) for r in x], 0.35, label='Good', color='#2ecc71', alpha=0.85)
ax2.bar(x + 0.2, [rating_bad.get(r, 0)  for r in x], 0.35, label='Bad',  color='#e74c3c', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'⭐{r}' for r in x])
ax2.set_title('Good/Bad phân theo Rating', fontweight='bold')
ax2.legend()

# 3. Confidence histogram
ax3 = fig.add_subplot(gs[0, 2])
reviews_df[reviews_df['label']=='Good']['confidence'].hist(bins=10, ax=ax3, color='#2ecc71', alpha=0.7, label='Good')
reviews_df[reviews_df['label']=='Bad']['confidence'].hist(bins=10, ax=ax3, color='#e74c3c', alpha=0.7, label='Bad')
ax3.set_title('Phân phối Confidence Score', fontweight='bold')
ax3.set_xlabel('Confidence')
ax3.legend()

# 4. Per-product analysis
product_names = dict(zip(products_raw['_id'].astype(str), products_raw['name']))
prod_agg = reviews_df.groupby('product_id').agg(
    total=('label','count'),
    good=('label', lambda x: (x=='Good').sum()),
    bad=('label', lambda x: (x=='Bad').sum()),
    avg_rating=('rating','mean'),
    avg_conf=('confidence','mean')
).reset_index()
prod_agg['name']       = prod_agg['product_id'].map(product_names)
prod_agg['good_ratio'] = prod_agg['good'] / prod_agg['total']
prod_agg = prod_agg.sort_values('good_ratio', ascending=True)

ax4 = fig.add_subplot(gs[1, :])
x_prod = np.arange(len(prod_agg))
ax4.barh(x_prod - 0.2, prod_agg['good'],  0.35, label='Good', color='#2ecc71', alpha=0.85)
ax4.barh(x_prod + 0.2, prod_agg['bad'],   0.35, label='Bad',  color='#e74c3c', alpha=0.85)
ax4.set_yticks(x_prod)
ax4.set_yticklabels(prod_agg['name'].str[:30], fontsize=8)
ax4.set_xlabel('Số lượng reviews')
ax4.set_title('Good vs Bad per Sản phẩm', fontweight='bold')
ax4.legend()

plt.suptitle('Sentiment Analysis – Furniture Shop Reviews', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('sentiment_analysis_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: sentiment_analysis_result.png')

## 7. Trend theo thời gian

In [ ]:
# Merge với created_at
merged = reviews_df.copy()
merged['createdAt'] = pd.to_datetime(reviews_raw['createdAt'].values, errors='coerce')
merged['date'] = merged['createdAt'].dt.strftime('%Y-%m-%d')

trend = (
    merged.dropna(subset=['date'])
    .groupby('date')
    .agg(
        reviews=('review_id','count'),
        avg_rating=('rating','mean'),
        good=('label', lambda x: (x=='Good').sum()),
        bad=('label', lambda x: (x=='Bad').sum()),
    )
    .reset_index()
    .sort_values('date')
)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Stacked bar: good/bad per ngày
axes[0].bar(trend['date'], trend['good'], label='Good', color='#2ecc71', alpha=0.85)
axes[0].bar(trend['date'], trend['bad'],  bottom=trend['good'], label='Bad', color='#e74c3c', alpha=0.85)
axes[0].set_title('Xu hướng Sentiment theo ngày', fontweight='bold')
axes[0].set_ylabel('Số reviews')
axes[0].legend()
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=7)

# Avg rating line
axes[1].plot(trend['date'], trend['avg_rating'], marker='o', color='#f39c12',
             linewidth=2, markersize=6, label='Avg Rating')
axes[1].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Neutral (3.0)')
axes[1].fill_between(trend['date'], 3, trend['avg_rating'],
                     where=trend['avg_rating'] >= 3, alpha=0.15, color='#2ecc71')
axes[1].fill_between(trend['date'], 3, trend['avg_rating'],
                     where=trend['avg_rating'] < 3, alpha=0.15, color='#e74c3c')
axes[1].set_ylim(1, 5.5)
axes[1].set_ylabel('Avg Rating')
axes[1].set_title('Điểm đánh giá trung bình theo ngày', fontweight='bold')
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

## 8. Xem toàn bộ kết quả phân tích

In [ ]:
# Kết quả chi tiết với product name
reviews_df['product_name'] = reviews_df['product_id'].map(product_names)

display_cols = ['product_name','rating','comment','label','confidence','score']
reviews_df[display_cols].sort_values(['label','confidence'], ascending=[True, False])

## 9. Tóm tắt luồng xử lý

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           Sentiment Analysis – Luồng Xử Lý                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  INPUT: reviews[{_id, product, user, rating, comment}]           ║
║                                                                  ║
║  sentiment_label(comment, rating):                               ║
║    1. text = comment.lower()                                     ║
║    2. score = Σ(hit_positive) - Σ(hit_negative)                 ║
║    3. if rating≥4 and score≥-1   → label='Good'                 ║
║       elif rating≤2 and score≤1  → label='Bad'                  ║
║       elif score > 0             → label='Good'                 ║
║       else                       → label='Bad'                  ║
║    4. confidence = min(1.0,                                      ║
║           0.55 + |score|×0.15 + |rating-3.0|×0.05)              ║
║                                                                  ║
║  sentiment_analysis(reviews):                                    ║
║    → Loop → {review_id, product_id, label, confidence, score}   ║
║    → summary: {total, good, bad, good_ratio, bad_ratio}         ║
║                                                                  ║
║  OUTPUT: {summary:{...}, reviews:[...]}                          ║
╚══════════════════════════════════════════════════════════════════╝
""")